In [ ]:
!pip install seaborn
!pip install holoviews
!pip install hvplot
!pip install altair

In [ ]:
# Our usual imports
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import geopandas as gpd
import holoviews as hv
import hvplot.pandas
import altair as alt


%matplotlib inline

# Matplotlib chart

You should consider what aspect of the dataset might be best plotted using matplotlib. Include your reasoning for using matplotlib to visualize this specific aspect of the data. You will be graded on the aesthetics of the plot, namely, color choices and clarity.

I select Parking violation from OpenDataPhilly.

In [ ]:
# merge two dataframes together to get the full year of parking violations data
parking_violations = pd.concat([
    pd.read_csv("./parking_violations_1st_half_2017.csv"),
    pd.read_csv("./parking_violations_2nd_half_2017.csv")
])
#reindex the dataframe to have a continuous index
parking_violations = parking_violations.reset_index(drop=True)
parking_violations.head()

In [ ]:
parking_violations = parking_violations.dropna(subset=["lat", "lon"])

In [ ]:
parking_violations["geometry"] = gpd.points_from_xy(
    parking_violations["lon"], parking_violations["lat"]
)

parking_violations["geometry"].head()

In [ ]:
parking_violations_gdf = gpd.GeoDataFrame(
    parking_violations, geometry="geometry", crs="EPSG:4326"
)

In [ ]:
# Convert the requested datetime to a column of Datetime objects
parking_violations_gdf["issue_datetime_dt"] = pd.to_datetime(
    parking_violations_gdf["issue_datetime"]
)

In [ ]:
parking_violations_gdf["issue_datetime_dt"].head()


In [ ]:
# Use the .dt attribute to extract out the month integer (starting at 0)
parking_violations_gdf["month_int"] = parking_violations_gdf["issue_datetime_dt"].dt.month

# Get the month name too — this is what %B means - (use strftime.org to see the codes!)
parking_violations_gdf["month"] = parking_violations_gdf["issue_datetime_dt"].dt.strftime("%B")

parking_violations_gdf.head()


In [ ]:
totals_by_month = parking_violations_gdf.groupby(
    by=["month_int", "month"], as_index=False
).size()

totals_by_month

In [ ]:
# Initialize figure/axes
fig, ax = plt.subplots(figsize=(12, 6))

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Plot!
ax.bar(
    totals_by_month["month_int"],
    totals_by_month["size"],
    color="#2176d2"
)

ax.set_xlabel("Month")
ax.set_ylabel("Number of Parking Violations")
# Set custom x-tick labels
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)

# Set axis labels
ax.set_xlabel("Months")
ax.set_ylabel("Number of Parking Violations")

# Start y-axis scale at 115,000
ax.set_ylim(bottom=115000)

# Add a title
ax.set_title(
    "Philadelphia's Parking Violations in 2017", weight="bold", fontsize=16
);

Total monthly violation counts were visualized using Matplotlib for its simple layout and easy formatting. It is able to highlight subtle seasonal changes, improving gridline transparency, and removing clutter. It can visualise the difference between each months. 

# Seaborn chart

Please include the motivation behind your choice for the type of seaborn plot used and why

In [ ]:
#showcase how many different violation codes there are in the dataset
len(parking_violations_gdf["violation_desc"].unique())

#showcase what violation codes are in the dataset
parking_violations_gdf["violation_desc"].unique()

In [ ]:
# Create categorized values in a separate column, preserving violation_desc
parking_violations_gdf['new_violation_desc'] = parking_violations_gdf['violation_desc'].copy()


# Map any violation that contains 'BLOCK' (covers both BLOCKING and BLOCKNG) to 'Blocking'
is_blocking = parking_violations_gdf['violation_desc'].str.contains('BLOCK', case=False, na=False)
parking_violations_gdf.loc[is_blocking, 'new_violation_desc'] = 'Blocking'

# Map any violation that contains 'METER' to 'Expired Meter'
is_expired = parking_violations_gdf['violation_desc'].str.contains('METER|MTR', case=False, na=False)
parking_violations_gdf.loc[is_expired, 'new_violation_desc'] = 'Expired Meter'

#Map any violation that contains 'STOP' to 'Illegal Stopping'
is_stopping = parking_violations_gdf['violation_desc'].str.contains('STOP', case=False, na=False)
parking_violations_gdf.loc[is_stopping, 'new_violation_desc'] = 'Illegal Stopping'

#Map any violation that contains 'PARK', or 'VALET' or 'KIOSK' or "reserved" or "loading" or "RES" to 'Illegal Parking'
is_parking = parking_violations_gdf['violation_desc'].str.contains('PARK|PK|VALET|KIOSK|RESERVED|LOADING|RES', case=False, na=False)
parking_violations_gdf.loc[is_parking, 'new_violation_desc'] = 'Illegal Parking'

#Map any violation that contains 'IMPROPER' to 'Improper Behavior'
is_improper = parking_violations_gdf['violation_desc'].str.contains('IMPROPER', case=False, na=False)
parking_violations_gdf.loc[is_improper, 'new_violation_desc'] = 'Improper Behavior'

# Map bus, taxi, and other public transit violations to 'Blocking Public Transit'
is_blocking_public_transit = parking_violations_gdf['violation_desc'].str.contains('BUS|TAXI|PUBLIC TRANSIT|TRANSIT', case=False, na=False)
parking_violations_gdf.loc[is_blocking_public_transit, 'new_violation_desc'] = 'Blocking Public Transit'

# Map sidewalk and pedestrian violations to 'Blocking Pedestrian'
is_blocking_pedestrian = parking_violations_gdf['violation_desc'].str.contains('SIDEWALK|PEDESTRIAN', case=False, na=False)
parking_violations_gdf.loc[is_blocking_pedestrian, 'new_violation_desc'] = 'Blocking Pedestrian'

#Map corner, pedestrian, intersection violations, and sidewalk violations to 'Blocking Pedestrian'
is_blocking_corner_pedestrian = parking_violations_gdf['violation_desc'].str.contains('CORNER|PEDESTRIAN|INTERSECTION|WALK', case=False, na=False)
parking_violations_gdf.loc[is_blocking_corner_pedestrian, 'new_violation_desc'] = 'Blocking Pedestrian'

# Map non-meter expiration violations to 'Expired Documentation'
is_expired_documentation = (
    parking_violations_gdf['violation_desc'].str.contains('EXPIRE|EXPIRATION', case=False, na=False)
    & ~is_expired
)
parking_violations_gdf.loc[is_expired_documentation, 'new_violation_desc'] = 'Expired Documentation'


#The remaining violation_desc values will be grouped into 'Other'
is_other = ~parking_violations_gdf['new_violation_desc'].isin(['Blocking', 'Blocking Public Transit', 'Blocking Pedestrian', 'Expired Meter', 'Expired Documentation', 'Illegal Stopping', 'Illegal Parking', 'Improper Behavior'])
parking_violations_gdf.loc[is_other, 'new_violation_desc'] = 'Other'

parking_violations_gdf


In [ ]:
violations_by_months = parking_violations_gdf.groupby(by=['month_int', 'month', 'new_violation_desc']).size().reset_index(name='count')

violations_by_months

In [ ]:
# --- Aesthetic Styling Setup ---
fig, ax = plt.subplots(figsize=(12, 6))

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

sns.histplot(
    data=violations_by_months,
    x="month_int",
    hue="new_violation_desc",
    weights="count",
    discrete=True,
    shrink=0.8,
    palette="Set1",  # Try "tab10", "Set2", "viridis", or "Spectral"
    multiple="stack",
    ax=ax,
)

# x/y axis labels
ax.set_xlabel("Months")
ax.set_ylabel("Violations")

ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)


# --- Move Legend Further Right and Rename Title ---
sns.move_legend(
    ax,
    loc="upper left",
    bbox_to_anchor=(1, 1),      # Increase 1.15 (e.g., to 1.25) to move it even further right
    title="Violation category",    # New legend title
    frameon=True                   # Keeps the background box around the legend
)

# Title
ax.set_title("Type of Philadelphia's Parking Violations in 2017", weight="bold", fontsize=16)

# Prevents the legend from being cut off at the edge of the image
plt.tight_layout()


I chose Seaborn's histplot() with multiple="stack" to visualize Philadelphia's monthly parking violations because it cleanly displays both overall ticket volume and category composition in a single chart. By stacking discrete monthly counts vertically instead of side-by-side, the plot avoids clutter while instantly highlighting dominant violation types—such as expired meters—and seasonal trends across 2017.

# 3 altair plots

### A transformation (mean, count, binning, etc)
### Brush selection Include a short discussion (a few sentences) of the main conclusion of each chart (in a markdown cell below each chart). It does not need to be interesting or insightful, but it is good practice to always note the main conclusions so the notebook make sense after time passes.

In [ ]:
parking_violations_gdf.head(5)

In [ ]:
parking_summary = (
    parking_violations_gdf
    .groupby(["month_int", "new_violation_desc"])
    .size()
    .reset_index(name="count")
)

parking_summary

In [ ]:
# Define a custom colormape using Hex codes & HTML color names
colormap = alt.Scale(
    domain=[0, 500, 1000, 2500, 5000, 7500, 10000, 20000, 50000],
    range=[
        "#F7FBFF",  # very light blue
        "#DEEBF7",
        "#C6DBEF",
        "#9ECAE1",
        "#6BAED6",
        "#4292C6",
        "#FDD835",  # yellow
        "#FB8C00",  # orange
        "#D32F2F",  # red
    ],
    type="sqrt",
)

In [ ]:
chart = (
    alt.Chart(parking_summary)
    .mark_rect()
    .encode(
        x=alt.X("month_int:N", axis=alt.Axis(title=None, ticks=False)),
        y=alt.Y("new_violation_desc:N", axis=alt.Axis(title=None, ticks=False)),
        color=alt.Color("count:Q", sort="ascending", scale=colormap),
        tooltip=["new_violation_desc", "month_int", "count"],
    )
    .properties(
        width=700,
        height=500,
        title="Parking Violations by Month and Type in Philadelphia 2017",
    )
    .configure_title(offset=20, fontWeight="bold")
)

chart

This heatmap shows that Philadelphia's parking violation volume remained remarkably stable across all twelve months of 2017, displaying minimal seasonal fluctuation. The Expired Meter category consistently generated the highest ticket volume in every single month, while physical obstruction categories like Blocking Pedestrian walkways and Blocking Public Transit maintained low, flat counts throughout the year. Overall, this demonstrates that enforcement efforts follow a steady, year-round routine dominated by routine meter monitoring rather than seasonal surges or localized obstruction responses.

In [ ]:
parking_location = (
    parking_violations_gdf
    .groupby(["new_violation_desc", "zip_code"])
    .size()
    .reset_index(name="count")
)

parking_location

# Heatmap by Zip Code

In [ ]:
import altair as alt

# Build the heatmap data in this cell so it does not depend on stale variables.
parking_location = (
    parking_violations_gdf
    .dropna(subset=['zip_code', 'new_violation_desc'])
    .assign(zip_code=lambda frame: frame['zip_code'].astype(int).astype(str))
    .groupby(['zip_code', 'new_violation_desc'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

# Limit the y-axis to the 15 ZIP codes with the most violations.
top_zips = (
    parking_location.groupby('zip_code')['count']
    .sum()
    .nlargest(15)
    .index
    .tolist()
)
df_filtered = parking_location[parking_location['zip_code'].isin(top_zips)]

alt.Chart(df_filtered).mark_rect().encode(
    x=alt.X(
        'new_violation_desc:N',
        title='Violation Category',
        axis=alt.Axis(labelAngle=-45),
    ),
    y=alt.Y('zip_code:N', title='ZIP Code', sort=top_zips),
    color=alt.Color(
        'count:Q',
        scale=alt.Scale(scheme='blues'),
        title='Ticket Count',
    ),
    tooltip=[
        alt.Tooltip('zip_code:N', title='ZIP Code'),
        alt.Tooltip('new_violation_desc:N', title='Category'),
        alt.Tooltip('count:Q', title='Ticket Count', format=','),
    ],
).properties(
    width=700,
    height=450,
    title='Philadelphia Parking Violations by ZIP Code and Category',
).interactive()

This heatmap reveals that parking violations in Philadelphia are heavily concentrated both geographically and by category. The darkest blue vertical band shows that Expired Meter violations generate the highest ticket volume across nearly all areas, with the greatest concentration occurring in Center City ZIP codes like 19103, 19107, 19147, and 19104. In contrast, safety and access infractions, such as Blocking Pedestrian walkways or Blocking Public Transit, have low ticket counts across every ZIP code. This reveals that ticketing is heavily concentrated on expired meters in downtown business districts, while safety violations like blocked sidewalks or driveways receive far less enforcement.

# Altair Map 3: Violations by Time of Day

In [ ]:
# Extract the hour and count violations by hour and category.
violations_by_hour = (
    parking_violations_gdf
    .dropna(subset=['issue_datetime_dt', 'new_violation_desc'])
    .assign(hour=lambda frame: frame['issue_datetime_dt'].dt.hour)
    .groupby(['hour', 'new_violation_desc'], as_index=False)
    .size()
    .rename(columns={'size': 'violations'})
)

category_order = sorted(violations_by_hour['new_violation_desc'].unique())
category_colors = {category: target_colors.get(category, '#6C757D') for category in category_order}

time_of_day_chart = (
    alt.Chart(violations_by_hour)
    .mark_line(point=True)
    .encode(
        x=alt.X('hour:Q', title='Time of Day (Hour)', scale=alt.Scale(domain=[0, 23]), axis=alt.Axis(values=list(range(24)))),
        y=alt.Y('violations:Q', title='Number of Violations'),
        color=alt.Color('new_violation_desc:N', title='Violation Category', scale=alt.Scale(domain=category_order, range=[category_colors[category] for category in category_order])),
        tooltip=[alt.Tooltip('hour:Q', title='Hour', format='02d'), alt.Tooltip('new_violation_desc:N', title='Category'), alt.Tooltip('violations:Q', title='Violations', format=',')],
    )
    .properties(width=900, height=500, title='Philadelphia Parking Violations by Time of Day and Category')
)

time_of_day_chart

This chart showcases the daily operational timeline and behavioral patterns of Philadelphia parking violations to reveal when enforcement activity peaks and what drives it. By plotting ticket volume across hours of the day broken down by category, the visualization proves that ticketing follows a strict diurnal business schedule—rising sharply at 8:00 AM, peaking around 12:00 PM, and dropping off significantly after standard business hours. Furthermore, it highlights that overall enforcement is overwhelmingly dominated by meter compliance rather than safety infractions, as the Expired Meter category vastly outpaces all other violation types during peak hours.

# Extra Credit - 2 Chart Dashboard

In [ ]:
# Build location/category counts from the source GeoDataFrame.
df_dashboard = (
    parking_violations_gdf
    .dropna(subset=['location', 'new_violation_desc'])
    .groupby(['location', 'new_violation_desc'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

# Filter to the 15 locations with the most violations so the dashboard remains clean.
top_locations = (
    df_dashboard.groupby('location')['count']
    .sum()
    .nlargest(15)
    .index
)
df_dashboard = df_dashboard[df_dashboard['location'].isin(top_locations)].copy()

# 2. Define the selection parameter for cross-filtering
# Clicking a location bar stores that selection
location_selection = alt.selection_point(fields=['location'])

# --- CHART 1: Top Parking Locations (The Filter Source) ---
chart1 = alt.Chart(df_dashboard).mark_bar().encode(
    x=alt.X('sum(count):Q', title='Total Violations'),
    y=alt.Y('location:N', sort='-x', title='Parking Location'),
    color=alt.condition(
        location_selection, 
        alt.value('#2176d2'),   # Selected bar turns blue
        alt.value('#d3d3d3')    # Unselected bars turn gray
    ),
    tooltip=['location', alt.Tooltip('sum(count):Q', format=',')]
).add_params(
    location_selection         # Attaches the interactive selection to Chart 1
).properties(
    width=350,
    height=400,
    title='1. Click a Location to Filter'
)

# --- CHART 2: Violation Category Breakdown (Cross-Filtered Target) ---
chart2 = alt.Chart(df_dashboard).mark_bar().encode(
    x=alt.X('sum(count):Q', title='Ticket Count'),
    y=alt.Y('new_violation_desc:N', sort='-x', title='Violation Category'),
    color=alt.Color('new_violation_desc:N', legend=None),
    tooltip=['new_violation_desc', alt.Tooltip('sum(count):Q', format=',')]
).transform_filter(
    location_selection         # Cross-filters Chart 2 based on selection in Chart 1
).properties(
    width=350,
    height=400,
    title='2. Violation Types at Selected Location'
)

# --- COMBINE INTO A 2-CHART DASHBOARD ---
dashboard = chart1 | chart2    # Combines both charts horizontally
dashboard